## Clustering - Unsupervised learning

In [31]:
import pandas as pd
import numpy as np
import joblib
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import davies_bouldin_score, make_scorer
import matplotlib.pyplot as plt

    - Loading data

In [32]:
data = pd.read_csv('../../data/traffic_data_cleaned_&_engineered.csv')

In [33]:
X_cluster = data[['traffic_volume', 'hour_sin', 'hour_cos', 'day_of_week']]

In [34]:
best_k = 4 # Default starting point

In [35]:
clustering_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('kmeans', KMeans(n_clusters=best_k, random_state=42, n_init=10))
])


In [36]:
data['traffic_cluster'] = clustering_pipeline.fit_predict(X_cluster)

In [37]:
db_index = davies_bouldin_score(clustering_pipeline.named_steps['scaler'].transform(X_cluster), data['traffic_cluster'])

In [38]:
print(f"Davies-Bouldin Index for K={best_k}: {db_index:.4f}")

Davies-Bouldin Index for K=4: 1.2248


In [39]:
k_values = [3, 4, 5, 6, 7]
init_methods = ['k-means++', 'random']
best_score = float('inf')
best_params = {}
best_clustering_pipeline = None

print("Starting Clustering Tuning...\n")

for k in k_values:
    for init in init_methods:
        # Create a Pipeline for this specific combination
        current_pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('kmeans', KMeans(n_clusters=k, init=init, n_init=10, random_state=42))
        ])
        
        # Fit and get labels
        # (We use X_cluster here so the pipeline handles the scaling)
        labels = current_pipeline.fit_predict(X_cluster)
        
        # Calculate Davies-Bouldin Index 
        # (We need to scale the data manually just for the scoring function)
        X_scaled_for_score = current_pipeline.named_steps['scaler'].transform(X_cluster)
        score = davies_bouldin_score(X_scaled_for_score, labels)
        
        print(f"K={k}, Init={init} -> DB Index: {score:.4f}")
        
        # Check if this is the best one so far
        if score < best_score:
            best_score = score
            best_params = {'k': k, 'init': init}
            best_clustering_pipeline = current_pipeline # Save the winning pipeline
print(f"\n--- Best Tuning Result ---")
print(f"Lowest DB Index: {best_score:.4f}")
print(f"Best Parameters: {best_params}")


Starting Clustering Tuning...

K=3, Init=k-means++ -> DB Index: 1.2172
K=3, Init=random -> DB Index: 1.2172
K=4, Init=k-means++ -> DB Index: 1.2248
K=4, Init=random -> DB Index: 1.2139
K=5, Init=k-means++ -> DB Index: 1.1032
K=5, Init=random -> DB Index: 1.1202
K=6, Init=k-means++ -> DB Index: 0.9532
K=6, Init=random -> DB Index: 0.9532
K=7, Init=k-means++ -> DB Index: 0.9541
K=7, Init=random -> DB Index: 0.9538

--- Best Tuning Result ---
Lowest DB Index: 0.9532
Best Parameters: {'k': 6, 'init': 'k-means++'}


In [40]:
joblib.dump(best_clustering_pipeline, '../../models/traffic_clustering_pipeline.joblib')
print("\nBest Clustering Pipeline saved successfully!")


Best Clustering Pipeline saved successfully!
